# Lab 9.12 &mdash; Capstone: The Financial-Report Insight Agent

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 5 &middot; Module 9 &mdash; Agents in Finance, Healthcare &amp; Cybersecurity**

### What you'll do
- Chain ground -> compute -> cite -> validate into one analyzer
- Reject any output with advice or an uncited claim
- Run it over a SUITE of reports and score it

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; grounding/citation/compute logic and, in the agent-assembly labs, tool wiring &amp; the read-only guardrail &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. You are building the **financial-report insight agent** &mdash; the client's Lab 5.1. It grounds &amp; cites every figure, gives **no advice**, and has **no trade tool** &mdash; a human analyst decides.

**Reference:** [Module 9 slides &mdash; The financial-report insight agent, end to end](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 9 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-09-12"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Capstone: the **financial-report insight agent** (the client's Lab 5.1), end to end. For each report
it **grounds** the figures, **computes** the metrics, builds a **cited** summary, **validates** that
every claim is cited and contains **no advice**, and returns it flagged **`needs_review`** for an
analyst &mdash; never a decision, never a trade. You run it over a **suite** of reports and score the
outcomes. The helpers below are the ones you built through the module.

In [ ]:
# A small financial report -- each figure carries its SOURCE, so every claim can be cited.
REPORT = {
    "revenue":    {"value": 120.0, "unit": "M", "source": "p4, income stmt"},
    "net_income": {"value": 9.0,   "unit": "M", "source": "p4, income stmt"},
    "total_debt": {"value": 40.0,  "unit": "M", "source": "p7, balance sheet"},
}
PRIOR = {"revenue": 107.1, "net_income": 9.7, "total_debt": 25.0}   # prior period, for YoY

# The pieces you built this module, provided so you can assemble the whole agent.
def margin_pct(ni, rev):
    return round(ni / rev * 100, 1)
def build_summary(report):
    rev, ni = report["revenue"], report["net_income"]
    m = margin_pct(ni["value"], rev["value"])
    return ("Revenue " + str(rev["value"]) + "M [" + rev["source"] + "]; "
            "net margin " + str(m) + "% [" + ni["source"] + "].")
def claims_of(report):
    return [{"metric": "revenue", "source": report["revenue"]["source"]},
            {"metric": "net_income", "source": report["net_income"]["source"]}]
ADVICE_TERMS = ("buy", "sell", "recommend", "you should", "invest in")
print("helpers ready: build_summary, claims_of, margin_pct")

## Your Turn
Assemble `process` (summary + cited + no-advice -> status) and `evaluate` (score the suite).

In [ ]:
def process(report):
    summary = build_summary(report)
    claims  = claims_of(report)
    cited   = ___   # TODO: True if every claim has a source
    advice  = any(t in summary.lower() for t in ADVICE_TERMS)
    # ship for review only if fully cited AND advice-free; else reject
    status  = "needs_review" if (___) else "rejected"   # TODO: cited and not advice
    return {"summary": summary, "cited": cited, "advice": advice, "status": status}

SUITE = [
    {"revenue": {"value": 120.0, "source": "p4"}, "net_income": {"value": 9.0, "source": "p4"}},
    {"revenue": {"value": 80.0,  "source": "p3"}, "net_income": {"value": 12.0, "source": "p3"}},
]

def evaluate():
    # a report is handled well if it is cited, advice-free, and flagged needs_review
    ok = ___   # TODO: count reports where cited and status == "needs_review"
    return ok, len(SUITE)

try:
    for r in SUITE:
        out = process(r)
        print(out['status'], '| cited:', out['cited'], '->', out['summary'][:52])
    print('score:', evaluate())
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("every figure in the summary is cited", lambda: process(SUITE[0])["cited"] is True)
expect_true("the summary contains no advice language", lambda: process(SUITE[0])["advice"] is False)
expect_true("the output is needs_review, never a decision or trade", lambda: process(SUITE[0])["status"] == "needs_review")
expect_true("a report becomes a cited summary", lambda: "p4" in process(SUITE[0])["summary"])
expect_true("an uncited/advice output would be rejected", lambda: process({"revenue": {"value": 1.0, "source": ""}, "net_income": {"value": 1.0, "source": "p1"}})["status"] == "rejected")
expect_true("the agent handles the whole suite well", lambda: evaluate() == (2, 2))

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run it for real (not graded)
Swap the offline pieces for a REAL model insight draft (Ollama / Groq) -- the bridge to Module 10. This calls a **real** local model via `ChatOllama("llama3.2:1b")` &mdash; start it with
`ollama run llama3.2:1b` (or swap in `ChatGroq` with a `GROQ_API_KEY`). If none is reachable the cell
prints a note and moves on. **The graded cells above never call an LLM, so the lab always verifies offline.**
*(llama3.2:1b is tiny &mdash; tool-calling can be hit-or-miss; the point is to see a real invocation.)*

In [ ]:
try:
    if ollama_up():
        from langchain_ollama import ChatOllama
        llm = ChatOllama(model="llama3.2:1b")
        out = llm.invoke("One-line, cite pages, NO advice: revenue 120M (p4), net income 9M (p4).").content
        print("REAL cited insight:\n", out)
        print("\nProduction shape: ground (tools/RAG) -> compute -> cite -> validate (grounded, no advice) ->")
        print("needs_review -> a human analyst decides. No trade tool, ever.")
    else:
        print("No Ollama reachable -- skipping the live draft. The offline insight agent above already ran the suite")
        print("(grounded, cited, advice-free, needs_review); no trade tool, ever.")
    print("Next: Module 10 -- ethics & responsible AI: bias, transparency, safety and accountability.")
except Exception as e:
    print("Live draft skipped:", type(e).__name__)

---
### Key takeaway
You built the financial-report insight agent end to end -- it grounds and cites every figure, gives no advice, has no trade tool, and flags for a human. That's a genuinely useful high-stakes agent. Next: Module 10 -- doing it responsibly.

[&#8592; All Module 9 labs](./index.html) &nbsp;&middot;&nbsp; [Module 9 slides](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>